# Fundamental Agent Prototype

This notebook is intentionally thin. All prompts, schemas, and reusable functions live in `src/openclam/agents/fundamental/fundamental_agent.py`, and the notebook only imports and calls those functions.

## 1. Environment Setup

Load the repo root, make `src/` importable, and load local environment variables from `.env` when available.

In [28]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

loaded_env = load_dotenv(REPO_ROOT / ".env", override=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded .env: {loaded_env}")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is not None:
    openai_key = userdata.get("OPENAI_API_KEY")
    fmp_key = userdata.get("FMP_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
    if fmp_key:
        os.environ["FMP_API_KEY"] = fmp_key

print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"FMP_API_KEY loaded: {bool(os.getenv('FMP_API_KEY'))}")


Repo root: C:\Users\22840\OneDrive\Desktop\genai_project\OpenClam_Multi-Agent_Investment_Advisory
Loaded .env: True
OPENAI_API_KEY loaded: True
FMP_API_KEY loaded: True


## 2. Import Fundamental Module

Reload the module during local iteration so notebook state does not hold stale definitions.

In [29]:
import importlib
import openclam.agents.fundamental.fundamental_agent as fundamental_agent

fundamental_agent = importlib.reload(fundamental_agent)

analyze_fundamental_input = fundamental_agent.analyze_fundamental_input
create_fundamental_agent = fundamental_agent.create_fundamental_agent
get_yfinance_fundamental_input = fundamental_agent.get_yfinance_fundamental_input
run_fundamental_analysis = fundamental_agent.run_fundamental_analysis
run_fundamental_workflow = fundamental_agent.run_fundamental_workflow

print(f"Loaded module from: {fundamental_agent.__file__}")


Loaded module from: C:\Users\22840\OneDrive\Desktop\genai_project\OpenClam_Multi-Agent_Investment_Advisory\src\openclam\agents\fundamental\fundamental_agent.py


## 3. Configure the Run

Set the ticker, transcript window, and model configuration used by the module helpers.

In [30]:
TICKER = "AAPL"
TRANSCRIPT_YEAR = 2025
TRANSCRIPT_QUARTER = 4
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.0


## 4. Create the Agent Client

This step builds the reusable agent object from the Python module.

In [31]:
agent = create_fundamental_agent(
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
)


## 5. Build Structured Fundamental Input

Generate the reusable structured input payload from market data and optional transcript context.

In [32]:
fundamental_input = get_yfinance_fundamental_input(
    TICKER,
    transcript_year=TRANSCRIPT_YEAR,
    transcript_quarter=TRANSCRIPT_QUARTER,
    require_transcript=True,
)

print(f"Transcript snippets loaded: {len(fundamental_input.earnings_call_snippets)}")
print(f"Guidance text available: {bool(fundamental_input.guidance_text)}")
print("===== AUTO-GENERATED INPUT =====")
print(fundamental_input.model_dump_json(indent=2))


Transcript snippets loaded: 20
Guidance text available: True
===== AUTO-GENERATED INPUT =====
{
  "ticker": "AAPL",
  "company_name": "Apple Inc.",
  "quarter": "2025-12-31",
  "earnings_summary": "\n    Company: Apple Inc.\n    Ticker: AAPL\n    Latest valid financial quarter: 2025-12-31\n\n    Important: current vs prior values below are quarter-over-quarter comparisons, NOT year-over-year.\n\n    Revenue QoQ: current=143756000000.0, previous_quarter=102466000000.0\n    Gross Profit QoQ: current=69231000000.0, previous_quarter=48341000000.0\n    Operating Income QoQ: current=50852000000.0, previous_quarter=32427000000.0\n    Net Income QoQ: current=42097000000.0, previous_quarter=27466000000.0\n    Operating Cash Flow QoQ: current=53925000000.0, previous_quarter=29728000000.0\n    Free Cash Flow QoQ: current=51552000000.0, previous_quarter=26486000000.0\n\n    Market Cap: 4110619508736.0\n    Trailing P/E: 33.92857\n    Forward P/E: 29.304062\n    Profit Margin: 0.27152002\n    Reven

## 6. Run the Fundamental Analysis

Pass the structured input into the module's analysis helper and print the final JSON output.

In [33]:
result = analyze_fundamental_input(
    fundamental_input,
    agent=agent,
)

print("\n===== FUNDAMENTAL AGENT OUTPUT =====")
print(result.model_dump_json(indent=2))



===== FUNDAMENTAL AGENT OUTPUT =====
{
  "ticker": "AAPL",
  "stance": "Bullish",
  "core_judgment": "Strong revenue growth and profitability with positive outlook for the upcoming quarter.",
  "positive_signals": [
    "Revenue increased significantly to $143.8 billion from $102.5 billion QoQ.",
    "Net income rose to $42.1 billion from $27.5 billion QoQ.",
    "Operating cash flow reached a record $53.9 billion, up from $29.7 billion QoQ.",
    "Services revenue set an all-time record of $28.8 billion, growing 15% YoY.",
    "Management expects December quarter revenue to grow by 10% to 12% YoY, indicating strong demand."
  ],
  "negative_signals": [
    "Operating expenses increased by 11% YoY, which could pressure margins.",
    "Gross margin guidance for December is between 47% and 48%, indicating potential cost pressures."
  ],
  "key_evidence": [
    "Management reported strong customer satisfaction and high levels of loyalty across product categories.",
    "iPhone revenue se

## 7. Optional Single-Call Workflow

Use this when you want both the generated input payload and the final model output from one module function.

```python
workflow_result = run_fundamental_workflow(
    TICKER,
    transcript_year=TRANSCRIPT_YEAR,
    transcript_quarter=TRANSCRIPT_QUARTER,
    require_transcript=True,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
)

print(workflow_result.model_dump_json(indent=2))
```